In [ ]:
from fastapi import APIRouter, HTTPException, status, Depends

from app.schemas.project import (
    ProjectCreate,
    ProjectUpdate,
    ProjectResponse,
)
from app.api.auth import get_current_user
from app.utils.helpers import generate_id, utc_now, record_activity


router = APIRouter(
    prefix="/projects",
    tags=["Projects"],
)


def get_database():
    """Return the application's MongoDB wrapper."""
    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


def project_to_response(project: dict) -> ProjectResponse:
    """Convert a MongoDB document into the API response model."""

    return ProjectResponse(
        id=project["id"],
        user_id=project["user_id"],
        space_id=project["space_id"],
        name=project["name"],
        description=project.get("description"),
        goal=project.get("goal"),
        created_at=project["created_at"],
        updated_at=project["updated_at"],
    )


def verify_space_ownership(
    database,
    space_id: str,
    user_id: str,
):
    """Verify that a Space exists and belongs to the authenticated user."""

    spaces = database.collection("spaces")

    space = spaces.find_one(
        {
            "id": space_id,
            "user_id": user_id,
        }
    )

    if space is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Space not found.",
        )

    return space


@router.post(
    "",
    response_model=ProjectResponse,
    status_code=status.HTTP_200_OK,
)
async def create_project(
    request: ProjectCreate,
    current_user=Depends(get_current_user),
):
    """Create a project inside an authorized Space."""

    database = get_database()

    # The Space must belong to the authenticated user.
    verify_space_ownership(
        database=database,
        space_id=request.space_id,
        user_id=current_user.id,
    )

    projects = database.collection("projects")

    now = utc_now()

    project = {
        "id": generate_id(),
        "user_id": current_user.id,
        "space_id": request.space_id,
        "name": request.name.strip(),
        "description": (
            request.description.strip()
            if request.description
            else None
        ),
        "goal": (
            request.goal.strip()
            if request.goal
            else None
        ),
        "created_at": now,
        "updated_at": now,
    }

    projects.insert_one(project)

    record_activity(
        database,
        user_id=current_user.id,
        project_id=project["id"],
        event_type="PROJECT_CREATED",
        description=f"Created project: {project['name']}",
        entity_type="project",
        entity_id=project["id"],
    )

    return project_to_response(project)


@router.get(
    "",
    response_model=list[ProjectResponse],
)
async def list_projects(
    current_user=Depends(get_current_user),
):
    """List projects visible to the authenticated user."""

    database = get_database()
    projects = database.collection("projects")

    documents = projects.find(
        {
            "user_id": current_user.id,
        }
    ).sort(
        "created_at",
        -1,
    )

    return [
        project_to_response(project)
        for project in documents
    ]


@router.get(
    "/{project_id}",
    response_model=ProjectResponse,
)
async def get_project(
    project_id: str,
    current_user=Depends(get_current_user),
):
    """Return an authorized project."""

    database = get_database()
    projects = database.collection("projects")

    project = projects.find_one(
        {
            "id": project_id,
            "user_id": current_user.id,
        }
    )

    if project is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Project not found.",
        )

    return project_to_response(project)


@router.put(
    "/{project_id}",
    response_model=ProjectResponse,
)
async def update_project(
    project_id: str,
    request: ProjectUpdate,
    current_user=Depends(get_current_user),
):
    """Update an authorized project."""

    database = get_database()
    projects = database.collection("projects")

    existing = projects.find_one(
        {
            "id": project_id,
            "user_id": current_user.id,
        }
    )

    if existing is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Project not found.",
        )

    update_data = {}

    if request.name is not None:
        update_data["name"] = request.name.strip()

    if request.description is not None:
        update_data["description"] = request.description.strip()

    if request.goal is not None:
        update_data["goal"] = request.goal.strip()

    update_data["updated_at"] = utc_now()

    projects.update_one(
        {
            "id": project_id,
            "user_id": current_user.id,
        },
        {
            "$set": update_data,
        },
    )

    updated = projects.find_one(
        {
            "id": project_id,
            "user_id": current_user.id,
        }
    )

    return project_to_response(updated)


@router.delete(
    "/{project_id}",
    status_code=status.HTTP_204_NO_CONTENT,
)
async def delete_project(
    project_id: str,
    current_user=Depends(get_current_user),
):
    """Delete an authorized project."""

    database = get_database()
    projects = database.collection("projects")

    result = projects.delete_one(
        {
            "id": project_id,
            "user_id": current_user.id,
        }
    )

    if result.deleted_count == 0:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Project not found.",
        )

    return None